# Class 8

- 📂 Convert raw Parquet/CSV data into Delta Tables using PySpark
- 🧪 Perform schema enforcement and time travel with versioned updates
- 🔄 Practice merge, upsert, and delete operations on Delta Tables
- 🏗️ Build a basic Medallion Architecture:
- Bronze ➝ raw ingestion
- Silver ➝ cleaned & filtered
- Gold ➝ ready for reporting
- 📌 Validate your pipeline using .history(), .describeDetail(), and other Delta tools

**1. What is Delta Lake?**

Key Concepts to Teach:

- Built on top of Parquet
- Supports ACID, schema enforcement, versioning
- Works best on Databricks and supports MERGE, UPDATE, DELETE

In [0]:
data = [("101", "Alice", "2023-11-01", 1000), ("102", "Bob", "2023-11-01", 2000)]
cols = ["order_id", "customer", "order_date", "amount"]

df = spark.createDataFrame(data,cols)
df.display()

In [0]:
#storing as a delta file
df.write.format("delta").mode("overwrite").save("/Volumes/data-bootcamp/default/all_csv/bronze_sales")

In [0]:
#now we want to display these file
display(dbutils.fs.ls("/Volumes/data-bootcamp/default/all_csv/bronze_sales"))

In [0]:
# this display will not work 
dbutils.fs.ls("/Volumes/data-bootcamp/default/all_csv/bronze_sales").display()

**display the delta logs**

In [0]:
display(dbutils.fs.ls("/Volumes/data-bootcamp/default/all_csv/bronze_sales/_delta_log"))


In [0]:
df_read = spark.read.format("delta") \
  .load("/Volumes/data-bootcamp/default/all_csv/bronze_sales")

df_read.display()


# Layers in Architecture (Medallion):
- Bronze ➝ Raw data
- Silver ➝ Cleaned + enriched data
- Gold ➝ Aggregated insights

## Build Medallion Architecture (Practical)
 🔶 Bronze – Ingest Raw CSV

In [0]:
# bronze layer
df_raw = spark.read.csv("/Volumes/data-bootcamp/default/all_csv/BigMart Sales.csv", header=True)
df_raw.write.format("delta").mode("overwrite").save("/Volumes/data-bootcamp/default/all_csv/bronze/sales")
display(df_raw)

In [0]:
# Silver Layer-clean and standarized
df_bronze = spark.read.format("delta").load("/Volumes/data-bootcamp/default/all_csv/bronze/sales")
df_silver = df_bronze.dropDuplicates()
df_silver.write.format("delta").mode("overwrite").save("/Volumes/data-bootcamp/default/all_csv/silver/sales")
display(df_silver)


**🟡 Gold – Aggregated Insights**



In [0]:
#here we just draw inside the silver layer
df_silver = spark.read.format("delta").load("/Volumes/data-bootcamp/default/all_csv/silver/sales")

df_gold = df_silver.groupBy("Item_Fat_Content").count()
df_gold.write.format("delta").mode("overwrite").saveAsTable("sales_count")
display(df.gold)

In [0]:
%sql
select * from sales_count

Databricks visualization. Run in Databricks to view.

In [0]:
# we can also add path to it

df_gold.write.format("delta") \
  .mode("overwrite") \
  .option("path", "/Volumes/data-bootcamp/default/all_csv/gold_sales_count") \
  .saveAsTable("sales_count1")


In [0]:
%sql
describe history sales_count